# KARAT: Knowledge-Aware Resilience via Adaptive Teacher-Student Distillation
## Full Experiment Walkthrough

This notebook reproduces all experiments from the paper:
- **E2**: Multi-seed main results (Table 2)
- **E5**: CIC-IDS2018 cross-domain validation
- **E6**: CKA vs L2 trigger ablation

### Setup
```bash
pip install -r requirements.txt
```


In [ ]:
import sys, os
sys.path.insert(0, '..')
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neural_network import MLPClassifier
from scipy.stats import wilcoxon
import warnings; warnings.filterwarnings('ignore')

from src.dataset import generate_kats_syn, KATS_FEATURES
from src.model import build_kats_ensemble
from src.attack import inject_adversarial
from src.metrics import compute_cka, compute_l2_div
from src.karat import score_karat, score_l2_trigger


## 1. Dataset and Model Setup

In [ ]:
SEED_MODEL = 42
K_FRAC     = 0.05
SCENARIOS  = ['S1_targeted', 'S2_coordinated', 'S3_cascading']
TIMESTEPS  = [0, 2, 4, 6, 8, 10]

df = generate_kats_syn(n=75000, seed=SEED_MODEL)
le = LabelEncoder(); le.fit(['Low', 'Medium', 'High'])
y  = le.transform(df['priority_label'])
HIGH_IDX = list(le.classes_).index('High')

idx = np.arange(len(df))
tr_i, te_i = train_test_split(idx, test_size=0.20, random_state=SEED_MODEL, stratify=y)
tr_i2, va_i = train_test_split(tr_i, test_size=0.25, random_state=SEED_MODEL, stratify=y[tr_i])
df_train = df.iloc[tr_i2].reset_index(drop=True)
df_val   = df.iloc[va_i].reset_index(drop=True)
df_test  = df.iloc[te_i].reset_index(drop=True)
y_train  = y[tr_i2]

teacher = build_kats_ensemble(alpha=5, seed=SEED_MODEL)
teacher.fit(df_train[KATS_FEATURES], y_train)

scaler_s = StandardScaler()
X_tr = scaler_s.fit_transform(df_train[KATS_FEATURES].values)
rng_kd = np.random.RandomState(SEED_MODEL)
kd_idx = rng_kd.choice(len(df_train), int(len(df_train)*0.08), replace=False)
student = MLPClassifier(hidden_layer_sizes=(16,), activation='relu',
                         max_iter=60, random_state=SEED_MODEL,
                         learning_rate_init=0.008, alpha=0.05)
student.fit(X_tr[kd_idx], y_train[kd_idx])

def teacher_proba(df_q): return teacher.predict_proba(df_q[KATS_FEATURES])
def student_proba(df_q): return student.predict_proba(scaler_s.transform(df_q[KATS_FEATURES].values))
def pak(df_q, sc):
    k = int(len(df_q)*K_FRAC); r = np.argsort(sc[:,HIGH_IDX])[::-1][:k]
    return (df_q['priority_label'].values[r]=='High').mean()

print(f'Teacher PAK@5%: {pak(df_test, teacher_proba(df_test)):.4f}')
print(f'Student PAK@5%: {pak(df_test, student_proba(df_test)):.4f}')


## 2. Threshold Calibration from Validation Split

In [ ]:
val_base_cka = compute_cka(teacher_proba(df_val), student_proba(df_val))
val_base_l2  = compute_l2_div(teacher_proba(df_val), student_proba(df_val))

val_cka_drops, val_l2_drops = [], []
for sc in SCENARIOS:
    for t in TIMESTEPS[1:]:
        df_tv = inject_adversarial(df_val, t, sc, SEED_MODEL)
        val_cka_drops.append(val_base_cka - compute_cka(teacher_proba(df_tv), student_proba(df_tv)))
        val_l2_drops.append(compute_l2_div(teacher_proba(df_tv), student_proba(df_tv)) - val_base_l2)

THETA     = max(float(np.percentile(val_cka_drops, 20)), 1e-5)
trig_rate = np.mean([d > THETA for d in val_cka_drops])
pos_l2    = [d for d in val_l2_drops if d > 0]
L2_THETA  = max(float(np.percentile(pos_l2, max(0,(1-trig_rate)*100))) if len(pos_l2)>=3
            else float(np.percentile(val_l2_drops, 50)), 1e-5)

BASE_CKA = compute_cka(teacher_proba(df_test), student_proba(df_test))
BASE_L2  = compute_l2_div(teacher_proba(df_test), student_proba(df_test))
print(f'THETA={THETA:.5f}  L2_THETA={L2_THETA:.6f}')
print(f'BASE_CKA={BASE_CKA:.4f}  BASE_L2={BASE_L2:.6f}')


## 3. E2 — Main Multi-Seed Experiment (Table 2)

In [ ]:
SEEDS = [42, 7, 13, 99, 2024]
all_runs = []
for seed in SEEDS:
    for scenario in SCENARIOS:
        for t in TIMESTEPS:
            df_t  = inject_adversarial(df_test, t, scenario, seed)
            tp    = teacher_proba(df_t); sp = student_proba(df_t)
            cka_d = BASE_CKA - compute_cka(tp, sp)
            l2_d  = compute_l2_div(tp, sp) - BASE_L2
            all_runs.append({
                'seed': seed, 'scenario': scenario, 'timestep': t,
                'PAK_B1':     round(pak(df_t, np.random.RandomState(seed).dirichlet(np.ones(3),len(df_t))), 4),
                'PAK_B2':     round(pak(df_t, sp), 4),
                'PAK_B3_L2':  round(pak(df_t, score_l2_trigger(sp, tp, l2_d, L2_THETA)), 4),
                'PAK_KARAT':  round(pak(df_t, score_karat(sp, tp, cka_d, THETA)), 4),
                'PAK_B4_ORC': round(pak(df_t, tp), 4),
            })

runs_df = pd.DataFrame(all_runs)
print('=== PAPER TABLE 2 ===')
for sc in SCENARIOS:
    sub = runs_df[runs_df['scenario']==sc]
    def ms(c): return f"{sub[c].mean():.4f}±{sub[c].std():.4f}"
    print(f"{sc:<22} B1={ms('PAK_B1')} B2={ms('PAK_B2')} ",
          f"B3={ms('PAK_B3_L2')} KARAT={ms('PAK_KARAT')} Orc={ms('PAK_B4_ORC')}")


## 4. E5 — CIC-IDS2018 Cross-Domain Validation

In [ ]:
# Load CIC dataset (must be preprocessed to match KATS_FEATURES schema)
# from src.dataset import load_cic_ids2018
# cic = load_cic_ids2018('data/CIC-IDS2018-preprocessed.csv')
# Then run: from experiments.E5_cic_validation import run_e5
# run_e5(teacher, student_proba, THETA, HIGH_IDX, cic)
print('See experiments/E5_cic_validation.py for the full run function.')


## 5. E6 — CKA vs L2 Ablation

In [ ]:
from experiments.E6_cka_vs_l2_ablation import run_e6
e6_df = run_e6(teacher_proba, student_proba, df_test,
               THETA, L2_THETA, BASE_CKA, BASE_L2, HIGH_IDX)
